<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# EarthDaily Agriculture — GDD Offset Extraction

Development notebook for the `GDDOffsetExtractor` class.

Exercises every entry point — `get_gdd_offset`, `get_gdd_offset_safe`, `format_gdd_offset_json`, `process_single_entity_gdd_offset`, and `process_entity_gdd_offset_bulk_parallel` — on a single entity before running bulk, mirroring the pattern used in `EDAgriculture_zoning_Function_Dev.ipynb`.

**Per-entity overrides:** `Date` (from `start_date`/`date` column) and `Offsets` (from `offsets` column) fall back to the extractor defaults when missing on the row.

## Step 1: Initialisation

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager

manager = WorkflowManager("prod", log_to_console=True, log_level="DEBUG")

## Step 2: Get entities

### Option 1 - Load entities from EarthDaily platform

In [ ]:
manager.load_seasonfields()
print(manager.sfd_list[['id', 'name']].head())

### Option 2 - Load entities from file

In [ ]:
# from earthdaily.agriculture.core.geometry import load_geodataframe
# manager.sfd_list = load_geodataframe("inputs/your_fields.parquet")
# print(f"Loaded {len(manager.sfd_list)} entities")
# print(manager.sfd_list.columns.tolist())

## Step 3: Configure extraction

In [ ]:
from earthdaily.agriculture.extractors.gdd_offset_functions import GDDOffsetExtractor

gdd_extractor = GDDOffsetExtractor(
    manager.bearer_token, manager.token_expiration, config=manager.config
)

# Params defaults — can be overridden per entity via start_date / offsets columns.
gdd_extractor.setup_gdd_offset_parameters(
    provider="GLOBAL1",
    lower_threshold=10,
    upper_threshold=20,
    date="2023-01-17",
    offsets=[100, 200],
)

### Build a single test entity

In [ ]:
test_entity = {
    "id": "test_001",
    "geometry": "POINT (-58.93681679 -13.72531769)",
    # Per-entity override examples (comment out to fall back to params):
    # "start_date": "2023-02-01",
    # "offsets": [50, 150, 300],
}
print(test_entity)

### Test API call

In [ ]:
print("--- Test: get_gdd_offset ---")
try:
    raw_response = gdd_extractor.get_gdd_offset(test_entity)
    print(f"Raw API response ({type(raw_response).__name__}):")
    print(raw_response)
except Exception as e:
    print(f"Error: {e}")

### Test safe API call

In [ ]:
print("--- Test: get_gdd_offset_safe ---")
safe_result = gdd_extractor.get_gdd_offset_safe(test_entity)
print(f"Success: {safe_result['success']}")
print(f"Error:   {safe_result['error']}")
print(f"Data:    {safe_result['data']}")

### Test format_gdd_offset_json

In [ ]:
print("--- Test: format_gdd_offset_json ---")
if safe_result['success'] and safe_result['data']:
    gdd_df = gdd_extractor.format_gdd_offset_json(safe_result['data'], entity_data=test_entity)
    print(f"Formatted DataFrame shape: {gdd_df.shape}")
    display(gdd_df)
else:
    print("No data to format")

### Test process_single_entity_gdd_offset

In [ ]:
import pandas as pd

row = pd.Series({
    "id": test_entity["id"],
    "name": "Test_Field",
    "geometry": test_entity["geometry"],
    # Uncomment to test per-entity overrides:
    # "start_date": "2023-02-01",
    # "offsets": "50,150,300",
})

result = gdd_extractor.process_single_entity_gdd_offset(row)
print(f"Error: {result['error']}")
if result['data'] is not None:
    print(f"Data shape: {result['data'].shape}")
    display(result['data'])

### Per-entity override test (entity carries date + offsets)

In [ ]:
row_override = pd.Series({
    "id": "test_override",
    "name": "Test_Field_Override",
    "geometry": "POINT (-58.93681679 -13.72531769)",
    "start_date": "2023-02-01",
    "offsets": [50, 150, 300],
})

result_override = gdd_extractor.process_single_entity_gdd_offset(row_override)
print(f"Error: {result_override['error']}")
if result_override['data'] is not None:
    print(f"Data shape: {result_override['data'].shape}")
    display(result_override['data'])

## Step 4: Bulk extraction

Run GDD offset on multiple entities in parallel. The entity DataFrame must expose `id` and `geometry`; `start_date` and `offsets` columns are optional per-entity overrides.

In [ ]:
# Build a small in-notebook entity DataFrame (replace with manager.sfd_list.head(N) for real runs).
entities = pd.DataFrame([
    {
        "id": "bulk_001",
        "geometry": "POINT (-58.93681679 -13.72531769)",
        # falls back to params.date and params.offsets
    },
    {
        "id": "bulk_002",
        "geometry": "POINT (-58.95000000 -13.73000000)",
        "start_date": "2023-02-01",
        "offsets": [50, 150, 300],
    },
])
entities

In [ ]:
bulk_results = gdd_extractor.process_entity_gdd_offset_bulk_parallel(
    entity_list=entities,
    max_workers=5,
    output_path=manager.output_result_dir,
    skip_export=False,
    prefix="gdd_offset",
)

print(f"Total: {bulk_results['total_calculations']}")
print(f"Successful: {bulk_results['successful_calculations']}")
print(f"Failed: {bulk_results['failed_calculations']}")
if not bulk_results['results_df'].empty:
    display(bulk_results['results_df'])

## Step 5: Spatial grouping (geohash dedup) + cache

GDD-offset is queried by field **centroid**, so fields whose centroids fall in
the same geohash cell **and** share the same base date + offsets + provider
return the same reached-dates. `spatial_grouping=True` calls the API **once per
`geohash × start_date × offsets × provider` group** and broadcasts to every
member field — same output shape as a per-field run.

- `spatial_precision` — geohash length (default `5` ≈ 4.9 km cells).
- Composes with `use_cache` (representative-unit cache stats).
- Result dict gains `representative_calls` / `grouped_from`.

In [ ]:
# Three fields: two co-located (same geohash-5 cell) + one ~12 km away (different cell).
demo_entities = pd.DataFrame([
    {"id": "grp_a1", "geometry": "POINT (-58.93681679 -13.72531769)"},
    {"id": "grp_a2", "geometry": "POINT (-58.93690000 -13.72540000)"},  # ~13 m from a1 → same cell
    {"id": "grp_b1", "geometry": "POINT (-58.85000000 -13.65000000)"},  # ~12 km away → different cell
])

per_field = gdd_extractor.process_entity_gdd_offset_bulk_parallel(
    entity_list=demo_entities, skip_export=True, spatial_grouping=False,
)
grouped = gdd_extractor.process_entity_gdd_offset_bulk_parallel(
    entity_list=demo_entities, skip_export=True,
    spatial_grouping=True, spatial_precision=5,
)

print(f"per-field API calls : {per_field['total_calculations']}")
print(f"grouped API calls   : {grouped['representative_calls']} (from {grouped['grouped_from']} fields)")
print(f"rows: per-field={len(per_field['results_df'])}  grouped={len(grouped['results_df'])}")
assert len(per_field['results_df']) == len(grouped['results_df']), "row-count invariant"
display(grouped['results_df'].head())

In [ ]:
# Cache cold vs warm (grouped + use_cache).
import time

for label in ("cold", "warm"):
    t0 = time.time()
    res = gdd_extractor.process_entity_gdd_offset_bulk_parallel(
        entity_list=demo_entities, skip_export=True,
        spatial_grouping=True, use_cache=True,
    )
    print(f"{label:>4} run: {time.time() - t0:5.2f}s  "
          f"calls={res['representative_calls']}  "
          f"cache_hit={res.get('cache_hit')}  cache_miss={res.get('cache_miss')}")